# Neural Networks from Scratch to Practice

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/03-deep-learning/02_neural_networks.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Understand feedforward networks deeply — activation functions, loss landscapes, optimizers, regularization, and hyperparameter tuning.

**Prerequisites:** PyTorch basics (notebook 01)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy matplotlib torch scikit-learn


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split

## 1. Activation Functions Visualized

In [ ]:
x = torch.linspace(-5, 5, 200)

activations = {
    'ReLU': torch.relu(x),
    'Sigmoid': torch.sigmoid(x),
    'Tanh': torch.tanh(x),
    'LeakyReLU': nn.LeakyReLU(0.1)(x),
    'GELU': nn.GELU()(x),
}

fig, axes = plt.subplots(1, len(activations), figsize=(18, 3))
for ax, (name, y) in zip(axes, activations.items()):
    ax.plot(x.numpy(), y.numpy(), linewidth=2)
    ax.set_title(name)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Non-Linear Decision Boundaries (make_moons)

In [ ]:
X_np, y_np = make_moons(n_samples=500, noise=0.2, random_state=42)
X_data = torch.FloatTensor(X_np)
y_data = torch.FloatTensor(y_np)

X_tr, X_te, y_tr, y_te = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

class MoonNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

model = MoonNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    y_pred = model(X_tr)
    loss = criterion(y_pred, y_tr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Plot decision boundary
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    zz = model(grid).numpy().reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, zz, levels=50, cmap='RdBu', alpha=0.8)
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap='RdBu', edgecolors='k', linewidth=0.5)
plt.title(f'Decision Boundary (Loss: {loss.item():.4f})')
plt.colorbar()
plt.show()

## 3. Loss Functions (MSE vs CrossEntropy)

In [ ]:
# Compare loss functions on a simple example
predictions = torch.linspace(0.01, 0.99, 100)
target_1 = torch.ones_like(predictions)

mse_loss = (predictions - target_1) ** 2
bce_loss = -(target_1 * torch.log(predictions))

plt.figure(figsize=(8, 5))
plt.plot(predictions.numpy(), mse_loss.numpy(), label='MSE Loss', linewidth=2)
plt.plot(predictions.numpy(), bce_loss.numpy(), label='BCE Loss', linewidth=2)
plt.xlabel('Predicted probability (true label = 1)')
plt.ylabel('Loss')
plt.title('MSE vs BCE when target = 1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print("BCE penalizes confident wrong predictions much more heavily than MSE")

## 4. Optimizers Compared (SGD, Adam, AdamW)

In [ ]:
torch.manual_seed(42)
X_opt, y_opt = make_moons(300, noise=0.2, random_state=42)
X_opt = torch.FloatTensor(X_opt)
y_opt = torch.FloatTensor(y_opt)

optimizer_configs = {
    'SGD (lr=0.1)': lambda p: optim.SGD(p, lr=0.1),
    'SGD+Momentum': lambda p: optim.SGD(p, lr=0.1, momentum=0.9),
    'Adam': lambda p: optim.Adam(p, lr=0.01),
    'AdamW': lambda p: optim.AdamW(p, lr=0.01, weight_decay=0.01),
}

histories = {}
for name, opt_fn in optimizer_configs.items():
    torch.manual_seed(42)
    model = MoonNet()
    optimizer = opt_fn(model.parameters())
    losses = []
    for epoch in range(200):
        y_pred = model(X_opt)
        loss = nn.BCELoss()(y_pred, y_opt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    histories[name] = losses

plt.figure(figsize=(10, 5))
for name, losses in histories.items():
    plt.plot(losses, label=name)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Optimizer Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Regularization (Dropout, Weight Decay, BatchNorm)

In [ ]:
class RegularizedNet(nn.Module):
    def __init__(self, use_dropout=False, use_batchnorm=False):
        super().__init__()
        layers = []
        layers.append(nn.Linear(2, 64))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(64))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(0.3))
        layers.append(nn.Linear(64, 32))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(32))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(0.3))
        layers.append(nn.Linear(32, 1))
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze()

configs = {
    'No regularization': {'use_dropout': False, 'use_batchnorm': False},
    'Dropout': {'use_dropout': True, 'use_batchnorm': False},
    'BatchNorm': {'use_dropout': False, 'use_batchnorm': True},
    'Both': {'use_dropout': True, 'use_batchnorm': True},
}

results = {}
for name, cfg in configs.items():
    torch.manual_seed(42)
    model = RegularizedNet(**cfg)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    losses = []
    for epoch in range(200):
        model.train()
        y_pred = model(X_opt)
        loss = nn.BCELoss()(y_pred, y_opt)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    results[name] = losses

plt.figure(figsize=(10, 5))
for name, losses in results.items():
    plt.plot(losses, label=name)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Effect of Regularization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Learning Rate Scheduling

In [ ]:
torch.manual_seed(42)
model = MoonNet()
optimizer = optim.Adam(model.parameters(), lr=0.01)

schedulers = {
    'StepLR': optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5),
    'CosineAnnealing': optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200),
}

# Visualize LR schedules
fig, ax = plt.subplots(figsize=(10, 4))
for name, scheduler_fn in [
    ('StepLR', lambda opt: optim.lr_scheduler.StepLR(opt, step_size=50, gamma=0.5)),
    ('CosineAnnealing', lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)),
    ('ExponentialLR', lambda opt: optim.lr_scheduler.ExponentialLR(opt, gamma=0.99)),
]:
    dummy_opt = optim.Adam([torch.randn(1, requires_grad=True)], lr=0.01)
    sched = scheduler_fn(dummy_opt)
    lrs = []
    for _ in range(200):
        lrs.append(dummy_opt.param_groups[0]['lr'])
        dummy_opt.step()
        sched.step()
    ax.plot(lrs, label=name)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('LR Schedule Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Try It Yourself

1. Train a 3-layer network on `make_circles` data. Visualize the decision boundary at epochs 10, 50, and 200.
2. Compare training with and without BatchNorm on a deeper network (5 layers). Plot both loss curves.
3. Implement early stopping: track validation loss and stop training when it hasn't improved for 10 epochs.

In [ ]:
# Your code here